# 32. 정보 있는 레벨에만 폴백

29번의 게이트는 `|SVR 예측 − 중앙값|` 하나만 본다. 모델이 근거를 못 찾았다고 판단하면
`mean_working` 폴백으로 되돌린다. 그런데 **폴백이 할 말이 있는지는 보지 않는다.**

`mean_working` 의 등위회귀 값을 보면 7~10 구간이 0.4657~0.4700 으로 전체 중앙값 0.48 과
사실상 같다. 이 레벨의 행을 폴백으로 되돌리는 건 "모르겠으니 중앙값"이라고 말하는 것과 같은데,
어중간하게 확신 있는 행까지 같이 끌려가면서 손해만 난다.

여기서는 **폴백 추정치가 중앙값과 충분히 다른 레벨에만** 폴백을 허용한다.

## 1. 설정

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, RobustScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
SEEDS = [42, 2024, 7, 123, 999, 2025]
NEW_SEEDS = [31337, 5, 2718, 1618]

NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']
CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
y = train['stress_score'].to_numpy(dtype=float)
print(train.shape, test.shape)

(3000, 18) (3000, 17)


## 2. 전처리와 모델

26·29번과 동일하다. `mean_working` 은 거리 계산에 넣지 않는다.

In [2]:
def numeric(df):
    x = df[NUM].copy()
    x['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(2)
    return x.to_numpy(dtype=float)

scaler = RobustScaler().fit(numeric(train))
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                    dtype=float).fit(train[CAT].fillna('Unknown'))

def build(df):
    return np.hstack([scaler.transform(numeric(df)),
                      ohe.transform(df[CAT].fillna('Unknown'))])

X, X_test = build(train), build(test)
LVL = train['mean_working'].round(0).fillna(-1).to_numpy(float)
LVL_TEST = test['mean_working'].round(0).fillna(-1).to_numpy(float)
MEDIAN = float(np.median(y))

def model():
    return TransformedTargetRegressor(
        regressor=SVR(C=4.0, gamma=2.0, kernel='rbf', epsilon=0.0),
        transformer=QuantileTransformer(output_distribution='normal',
                                        n_quantiles=1000, random_state=RANDOM_STATE))

def iso_table(lv_tr, y_tr, lv_target):
    m = lv_tr >= 0
    ir = IsotonicRegression(out_of_bounds='clip').fit(lv_tr[m], y_tr[m])
    return np.where(lv_target >= 0,
                    ir.predict(np.where(lv_target >= 0, lv_target, 0)),
                    y_tr.mean()).astype(float)

print(f'X {X.shape}, 타겟 중앙값 {MEDIAN}')

X (3000, 32), 타겟 중앙값 0.48


## 3. 레벨별로 폴백이 할 말이 있는가

등위회귀 추정치가 중앙값에서 얼마나 떨어져 있는지 본다.
이 값이 작으면 그 레벨의 폴백은 "중앙값"이라는 말밖에 못 한다.

In [3]:
lv_all = np.r_[np.arange(4., 17.), -1.]
iso_all = iso_table(LVL, y, lv_all)
info = pd.DataFrame({
    'n': [int((LVL == v).sum()) for v in lv_all],
    '등위회귀': iso_all.round(4),
    '|중앙값과 차이|': np.abs(iso_all - MEDIAN).round(4),
}, index=[int(v) if v > 0 else '결측' for v in lv_all])
info.index.name = 'mean_working'
print(info.to_string())

gap_lo = np.abs(iso_all - MEDIAN)[np.abs(iso_all - MEDIAN) < 0.05].max()
gap_hi = np.abs(iso_all - MEDIAN)[np.abs(iso_all - MEDIAN) > 0.05].min()
print(f'\n두 무리로 갈린다: 아래쪽 최대 {gap_lo:.4f}, 위쪽 최소 {gap_hi:.4f} (약 {gap_hi/gap_lo:.0f}배)')

                 n    등위회귀  |중앙값과 차이|
mean_working                         
4                5  0.2780     0.2020
5               20  0.3055     0.1745
6               94  0.3148     0.1652
7              318  0.4657     0.0143
8              451  0.4700     0.0100
9              537  0.4700     0.0100
10             346  0.4700     0.0100
11             120  0.5964     0.1164
12              26  0.7236     0.2436
13              23  0.7236     0.2436
14               9  0.7236     0.2436
15              17  0.7236     0.2436
16               2  0.7236     0.2436
결측            1032  0.4821     0.0021

두 무리로 갈린다: 아래쪽 최대 0.0143, 위쪽 최소 0.1164 (약 8배)


7~10 과 결측이 한 무리(0.0021~0.0143), 나머지가 다른 무리(0.1164~0.2436)다.
사이가 8배 넘게 비어 있어서 **어디를 잘라도 같은 분할**이 나온다.
조정할 파라미터가 아니라 데이터가 알려주는 경계다.

`mean_working` 은 양 끝에서만 정보를 준다. 가운데 구간은 전체 중앙값과 구분되지 않는다.

## 4. 게이트

29번 게이트에 조건을 하나 더 곱한다.

`w = exp(-(|예측 − 중앙값| / tau)^2) × [ |폴백 − 중앙값| > THR ]`

앞항은 "모델이 근거를 못 찾았다", 뒷항은 "폴백이 할 말이 있다"다. 둘 다 참일 때만 폴백한다.

폴백 추정기도 29번의 레벨별 EB 축소 평균에서 등위회귀로 바꿨다.
레벨표는 각 레벨을 독립으로 보지만, 등위회귀는 단조성을 가정해 인접 레벨끼리 표본을 공유한다.
표본이 적은 레벨(mw=16, n=2)이 전체 평균이 아니라 이웃 레벨 쪽으로 당겨진다.

바뀐 것이 둘이므로 아래에서 단계별로 나눠 잰다.

In [4]:
TAU, THR = 0.03, 0.04
EB_K = 20

def eb_table(lv_tr, y_tr, lv_target, k=EB_K):
    """29번이 쓰던 레벨별 경험적 베이즈 축소 평균."""
    g = pd.DataFrame({'l': lv_tr, 'y': y_tr}).groupby('l')['y'].agg(['count', 'mean'])
    gm = y_tr.mean()
    eb = (g['count'] * g['mean'] + k * gm) / (g['count'] + k)
    return pd.Series(lv_target).map(eb).fillna(gm).to_numpy(float)

def oof(seed):
    ps = np.zeros(len(y)); fb = np.zeros(len(y)); fb_eb = np.zeros(len(y))
    for t, v in KFold(5, shuffle=True, random_state=seed).split(X):
        ps[v] = np.clip(model().fit(X[t], y[t]).predict(X[v]), 0, 1)
        fb[v] = iso_table(LVL[t], y[t], LVL[v])
        fb_eb[v] = eb_table(LVL[t], y[t], LVL[v])
    return ps, fb, fb_eb

def blend_plain(ps, fb, tau=0.02):
    """29번 방식 게이트. 폴백 추정기만 인자로 갈아끼운다."""
    w = np.exp(-((np.abs(ps - MEDIAN) / tau) ** 2))
    return np.clip((1 - w) * ps + w * fb, 0, 1)

def blend_info(ps, fb, tau=TAU, thr=THR):
    """32번. 폴백이 중앙값과 충분히 다른 레벨에만 게이트를 연다."""
    w = np.exp(-((np.abs(ps - MEDIAN) / tau) ** 2)) * (np.abs(fb - MEDIAN) > thr)
    return np.clip((1 - w) * ps + w * fb, 0, 1)

## 5. 검증

시드 10개로 대응비교한다. 뒤의 4개는 이번에 처음 쓰는 시드다.
같은 시드·같은 폴드·같은 SVR 예측 위에서 두 변경을 차례로 얹는다.

In [5]:
rows = []
for s in SEEDS + NEW_SEEDS:
    ps, fb, fb_eb = oof(s)
    rows.append({'시드': s, '구분': '기존' if s in SEEDS else '신규',
                 'SVR 단독': mean_absolute_error(y, ps),
                 '29번 EB': mean_absolute_error(y, blend_plain(ps, fb_eb)),
                 '+등위회귀': mean_absolute_error(y, blend_plain(ps, fb)),
                 '+정보 게이트': mean_absolute_error(y, blend_info(ps, fb))})

res = pd.DataFrame(rows)
res['등위회귀 효과'] = res['+등위회귀'] - res['29번 EB']
res['게이트 효과'] = res['+정보 게이트'] - res['+등위회귀']
res['29번 대비'] = res['+정보 게이트'] - res['29번 EB']
print(res.round(6).to_string(index=False))
print(f"\n등위회귀 효과 {res['등위회귀 효과'].mean():+.6f}  "
      f"부호 일관 {bool((res['등위회귀 효과'] < 0).all())}")
print(f"게이트 효과   {res['게이트 효과'].mean():+.6f}  "
      f"부호 일관 {bool((res['게이트 효과'] < 0).all())}")
print(f"29번 대비 합계 {res['29번 대비'].mean():+.6f}")
print(f"상수 {MEDIAN} 기준선 {mean_absolute_error(y, np.full_like(y, MEDIAN)):.6f}")

   시드 구분   SVR 단독   29번 EB    +등위회귀  +정보 게이트   등위회귀 효과    게이트 효과    29번 대비
   42 기존 0.147668 0.145707 0.145255 0.145008 -0.000452 -0.000248 -0.000700
 2024 기존 0.144505 0.142118 0.141624 0.141411 -0.000493 -0.000213 -0.000707
    7 기존 0.146518 0.144733 0.144157 0.143736 -0.000576 -0.000421 -0.000997
  123 기존 0.145962 0.144175 0.143756 0.143301 -0.000418 -0.000456 -0.000874
  999 기존 0.145809 0.143690 0.143136 0.142705 -0.000554 -0.000431 -0.000985
 2025 기존 0.149742 0.147235 0.146591 0.146294 -0.000644 -0.000297 -0.000941
31337 신규 0.147193 0.145246 0.144845 0.144523 -0.000402 -0.000322 -0.000723
    5 신규 0.146467 0.144444 0.143896 0.143557 -0.000549 -0.000339 -0.000887
 2718 신규 0.148250 0.146283 0.145591 0.145340 -0.000692 -0.000251 -0.000943
 1618 신규 0.145674 0.143757 0.143178 0.143062 -0.000580 -0.000116 -0.000696

등위회귀 효과 -0.000536  부호 일관 True
게이트 효과   -0.000309  부호 일관 True
29번 대비 합계 -0.000845
상수 0.48 기준선 0.249443


두 변경 모두 시드 10개 전부에서 개선됐다. 폴드 분할 과적합이 아니다.

28번의 GroupKFold(근접 중복행을 한 폴드로 묶어 중복 효과를 제거한 검증)로 그룹 순열을 바꿔가며
8번 재면 등위회귀까지 적용한 상태가 0.245537, 정보 게이트까지 적용하면 **0.245448** 로
8분할 중 7개에서 개선된다. 상수 기준선은 0.249443 이다.
**중복을 걷어내도 이득이 남는다.** 해당 검증 코드는 쌍 탐지를 포함하므로 28번에 두고
제출 노트북에는 넣지 않았다.

## 6. THR 고원과 TAU 선택

`THR` 이 튜닝된 값이 아니라 분할 경계라는 걸 보이고, `TAU` 를 무엇으로 골랐는지 적어 둔다.

In [6]:
ps, fb, _ = oof(42)

print('THR (TAU=0.03 고정)')
print(f'{"THR":>8}{"MAE":>12}{"폴백 허용 행":>14}')
for thr in [0.015, 0.02, 0.04, 0.06, 0.08, 0.10, 0.11, 0.13]:
    m = mean_absolute_error(y, blend_info(ps, fb, thr=thr))
    print(f'{thr:>8.3f}{m:>12.6f}{int((np.abs(fb - MEDIAN) > thr).sum()):>14}')
print('0.04~0.10 구간에서 값이 완전히 같다. 그 안에서는 어디를 잘라도 같은 레벨 분할이다.')

print('\nTAU (THR=0.04 고정)')
print(f'{"TAU":>8}{"MAE":>12}')
for tau in [0.01, 0.02, 0.03, 0.04, 0.06, 0.09]:
    print(f'{tau:>8.3f}{mean_absolute_error(y, blend_info(ps, fb, tau=tau)):>12.6f}')

THR (TAU=0.03 고정)
     THR         MAE       폴백 허용 행
   0.015    0.145108           712
   0.020    0.145034           381
   0.040    0.145008           316
   0.060    0.145008           316
   0.080    0.145008           316
   0.100    0.145008           316
   0.110    0.145350           272
   0.130    0.145653           196
0.04~0.10 구간에서 값이 완전히 같다. 그 안에서는 어디를 잘라도 같은 레벨 분할이다.

TAU (THR=0.04 고정)
     TAU         MAE
   0.010    0.145508
   0.020    0.144995
   0.030    0.145008
   0.040    0.145114
   0.060    0.145374
   0.090    0.145728


`THR` 은 고원이라 고민할 것이 없지만 `TAU` 는 다르다. **KFold 로만 보면 0.02 가 0.03 보다
약 0.0001 낮다.** 그런데 28번의 GroupKFold 로 재면 0.02 가 0.245622, 0.03 이 **0.245390** 으로
0.03 만 "중복을 걷어내도 이득이 남는다"는 조건을 만족한다.

차이가 0.0001 인 구간이라 **일반화되는 쪽을 택해 0.03 을 썼다.** 두 값 모두 상수 기준선
0.249443 은 이긴다. 리더보드만 보면 0.02 가 근소하게 유리할 수 있다는 점은 밝혀 둔다.

## 7. 최종 학습 및 제출

In [7]:
final_svr = model().fit(X, y)
pred_svr = np.clip(final_svr.predict(X_test), 0, 1)
pred_fb = iso_table(LVL, y, LVL_TEST)
pred = blend_info(pred_svr, pred_fb)

allow = np.abs(pred_fb - MEDIAN) > THR
w = np.exp(-((np.abs(pred_svr - MEDIAN) / TAU) ** 2)) * allow
print(f'폴백이 허용된 행   : {int(allow.sum())}개 ({allow.mean()*100:.1f}%)')
print(f'실제로 이동한 행   : {int((w > 0.5).sum())}개')
print(f'평균 이동폭       : {np.abs(pred - pred_svr).mean():.4f}')
print(f'29번 대비 예측 변화 : '
      f'{np.abs(pred - blend_plain(pred_svr, eb_table(LVL, y, LVL_TEST))).mean():.4f}')
print(f'예측 평균 {pred.mean():.4f}, 표준편차 {pred.std():.4f}, '
      f'범위 {pred.min():.3f}~{pred.max():.3f}')

sub = pd.read_csv('../data/sample_submission.csv')
sub['stress_score'] = pred
sub.to_csv('../submissions/submit_32_informative_gate.csv', index=False)
print('\nsaved -> submissions/submit_32_informative_gate.csv')
print(sub.head().to_string(index=False))

폴백이 허용된 행   : 318개 (10.6%)
실제로 이동한 행   : 183개
평균 이동폭       : 0.0092
29번 대비 예측 변화 : 0.0084
예측 평균 0.5004, 표준편차 0.2032, 범위 0.000~1.000

saved -> submissions/submit_32_informative_gate.csv
       ID  stress_score
TEST_0000          0.49
TEST_0001          0.97
TEST_0002          0.19
TEST_0003          0.49
TEST_0004          0.53


## 8. 정리

| 항목 | 29번 | 32번 |
|---|---|---|
| SVR | C=4.0 gamma=2.0 | 동일 |
| 폴백 추정기 | 레벨별 EB 축소 평균 (k=20) | 등위회귀 |
| 폴백 조건 | 예측이 중앙값 근처 | + 폴백이 중앙값과 다를 것 |
| CV (시드 10개) | 0.144739 | 0.143894 |
| GroupKFold (8분할) | — | 0.245448 |

29번은 `mean_working` 에 신호가 있다는 걸 찾았다.
32번은 **그 신호가 어디에 없는지**를 쓴다. 7~10 구간과 결측(2684행)에서 폴백은 중앙값을
다시 말하는 것뿐이라, 거기에 폴백을 거는 건 어중간하게 확신 있는 예측만 깎는다.
폴백이 실제로 걸리는 행은 3000행 중 316행뿐이다.

`THR` 은 0.04~0.10 어디로 두어도 같은 결과다. 레벨들이 두 무리로 8배 넘게 벌어져 있어서
그 사이 아무 데나 자르면 된다. 조정할 파라미터가 아니라 데이터가 알려주는 경계다.

### 규정 관련

- 스케일러·인코더·등위회귀 전부 train(또는 학습 폴드)에서만 적합하고 test 에는 적용만 했다
- 근접 중복행 탐색, 최근접이웃 매칭, test 행 간 정보 공유 코드가 없다
- 게이트 두 항 모두 모델 자기 출력과 train 레벨 통계만 쓴다